In [ ]:
import torch
import torch.nn as nn
import timm
import torchvision.transforms as transforms
from PIL import Image
from flask import Flask, request, jsonify
import io
import base64
import geopandas as gpd
from shapely.geometry import Polygon, Point
import json
import random

app = Flask(__name__)
# fmt: off
country_names = ["Albania", "Andorra", "Antarctica", "Argentina", "Australia", "Austria", "Bangladesh", "Belgium", "Bermuda", "Bhutan", "Bolivia", "Botswana", "Brazil", "Bulgaria", "Cambodia", "Canada", "Chile", "Australia", "Australia", "Colombia", "Costa Rica", "Croatia", "Netherlands", "Czechia", "Denmark", "Dominican Republic", "Ecuador", "Estonia", "Eswatini", "Faroe Islands", "Finland", "France", "Germany", "Ghana", "Gibraltar", "Greece", "Denmark", "United States", "Guatemala", "China", "Hungary", "Iceland", "India", "Ireland", "United Kingdom", "Israel", "Italy", "Japan", "UnitedKingdom", "Jordan", "Kazakhstan", "Kenya", "Kyrgyzstan", "Laos", "Latvia", "Lebanon", "Lesotho", "Liechtenstein", "Lithuania", "Luxembourg", "Malta", "Mexico", "Monaco", "Mongolia", "Montenegro", "Namibia", "Nepal", "Netherlands", "New Zealand", "Nigeria", "North Macedonia", "United States", "Norway", "Oman", "Pakistan", "Panama", "Peru", "Philippines", "Poland", "Portugal", "United States", "Qatar", "Romania", "Rwanda", "France", "San Marino", "Senegal", "Serbia", "Singapore", "Slovakia", "Slovenia", "South Africa", "South Korea", "Spain", "Sri Lanka", "Sweden", "Switzerland", "Taiwan", "Thailand", "Tunisia", "Turkey", "United States", "Uganda", "Ukraine", "United Arab Emirates", "United States", "Vietnam", "Indonesia", "Malaysia", "Russia", "United Kingdom", "Uruguay",
]
# fmt: on
NUM_COUNTRY_CLASSES = len(country_names)
NUM_SQUARE_CLASSES = 3734
gdf = gpd.read_file("countryBoundaries.geojson")

with open("label_mapping (3.5k).json", "r") as f:
    sqaureLabels = json.load(f)

finalCoords = {"lng": 0.0, "lat": 0.0}


# CORS headers for all responses
@app.after_request
def after_request(response):
    response.headers.add("Access-Control-Allow-Origin", "*")
    response.headers.add("Access-Control-Allow-Headers", "Content-Type,Authorization")
    response.headers.add("Access-Control-Allow-Methods", "GET,PUT,POST,DELETE,OPTIONS")
    return response


# model architecture for country
class CountryCustomTinyViT(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        in_features = backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.classifier(x)
        return x


# model architecture for square
class SquareCustomTinyViT(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        in_features = backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.classifier(x)
        return x


try:
    print(f"Loading model...")

    backbone_country = timm.create_model(
        "tiny_vit_21m_224", pretrained=True, num_classes=0
    )
    backbone_square = timm.create_model(
        "tiny_vit_5m_224", pretrained=True, num_classes=0
    )

    countryModel = CountryCustomTinyViT(backbone_country, NUM_COUNTRY_CLASSES)
    squareModel = SquareCustomTinyViT(backbone_square, NUM_SQUARE_CLASSES)

    # Load the state dictionary
    countryModelState = torch.load("mid_weights.pth", map_location="cpu")
    squareModelState = torch.load(
        "tinyvit_squares_3st_GELU_AdamW_3.4klabels.pth", map_location="cpu"
    )

    # Check if the state dict matches the model architecture
    try:
        countryModel.load_state_dict(countryModelState)
        squareModel.load_state_dict(squareModelState)

    except RuntimeError as e:
        print(f"Error loading state dict: {e}")
        raise

    countryModel.eval()
    squareModel.eval()
    print(f"Models loaded successfully")
except Exception as e:
    print(f"Error loading model: {e}")
    raise

# match training transformations
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)


def is_square_in_country(square_id, country_id):

    try:
        country = country_names[country_id]
        # print(f"  Looking for country: {country}")

        country_row = gdf[gdf["shapeName"] == country]

        # Get the first geometry (assuming one country per name)
        country_geometry = country_row.geometry.iloc[0]
        # print(f"  Country geometry type: {type(country_geometry)}")

        # all coord labels are in style "30.0_40.0_50.0_60.0" for min_lat,max_lat,min_lon,max_lon
        square_coords_str = sqaureLabels.get(str(square_id))
        if not square_coords_str:
            print(f"  Square ID {square_id} not found in mapping")
            return False

        square_coord_array = square_coords_str.split("_")

        min_lat, max_lat, min_lon, max_lon = map(float, square_coord_array)
        # print(f"  Square bounds: ({min_lat}, {max_lat}) to ({min_lon}, {max_lon})")

        square_coords = [
            (min_lon, max_lat),  # top-left
            (max_lon, max_lat),  # top-right
            (max_lon, min_lat),  # bottom-right
            (min_lon, min_lat),  # bottom-left
            (min_lon, max_lat),  # close the polygon
        ]

        square_polygon = Polygon(square_coords)

        # Check if the square intersects with the country
        intersects = country_geometry.intersects(square_polygon)
        if intersects:
            global finalCoords
            # Update final coordinates to the center of the square
            finalCoords["lng"] = (min_lon + max_lon) / 2
            finalCoords["lat"] = (min_lat + max_lat) / 2
            print(
                f" Min lat: {min_lat}, Max lat: {max_lat}, Min lon: {min_lon}, Max lon: {max_lon}"
            )
            finalCoords["lat"], finalCoords["lng"] = is_coords_in_country(
                finalCoords["lat"],
                finalCoords["lng"],
                country_geometry,
                min_lat,
                max_lat,
                min_lon,
                max_lon,
            )

            print(f"  Updated final coordinates: {finalCoords}")

        return intersects

    except Exception as e:
        print(f"  Error in is_square_in_country: {e}")
        import traceback

        traceback.print_exc()
        return False


def is_coords_in_country(
    lat, lng, country_geometry, min_lat, max_lat, min_lon, max_lon, attempts=0
):
    if attempts >= 999:
        print("Max attempts reached, returning random coordinates")
        return lat, lng
    point = Point(lng, lat)
    if country_geometry.contains(point):
        return lat, lng
    else:
        new_lat = random.uniform(min_lat, max_lat)
        new_lng = random.uniform(min_lon, max_lon)
        return is_coords_in_country(
            new_lat, new_lng, country_geometry, min_lat, max_lat, min_lon, max_lon, attempts + 1
        )


def prediction(input_tensor):
    with torch.no_grad():

        country_output = countryModel(input_tensor)
        square_output = squareModel(input_tensor)

        # single country prediction
        country_prediction = torch.argmax(country_output, dim=1).item()

        # many square predictions sorted by confidence
        square_probs = torch.softmax(square_output, dim=1)
        sorted_square_indices = torch.argsort(
            square_probs, dim=1, descending=True
        ).squeeze(0)

        print(
            f"Country prediction: {country_prediction} ({country_names[country_prediction]})"
        )
        print(f"Top 5 square predictions: {sorted_square_indices[:5].tolist()}")

        # Checks for highest confidence square prediction that is valid
        valid_square_prediction = None
        checked_squares = 0

        for square_idx in sorted_square_indices[:NUM_SQUARE_CLASSES]:
            square_candidate = square_idx.item()
            checked_squares += 1

            if is_square_in_country(square_candidate, country_prediction):
                valid_square_prediction = square_candidate
                print(
                    f"Found valid square: {square_candidate} after checking {checked_squares} squares"
                )
                break
            
    return finalCoords["lat"], finalCoords["lng"]

  



In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/dev/GeoguessrAI/testSetImageStitched.zip'
extract_path = '/content/unzipped'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Unzipped to:", extract_path)

In [ ]:
CLASS_NAMES = {
    0: "Albania", 1: "Andorra", 2: "Antarctica", 3: "Argentina",
    4: "Australia", 5: "Austria", 6: "Bangladesh", 7: "Belgium",
    8: "Bermuda", 9: "Bhutan", 10: "Bolivia", 11: "Botswana",
    12: "Brazil", 13: "Bulgaria", 14: "Cambodia", 15: "Canada",
    16: "Chile", 17: "Christmas Island", 18: "Cocos (Keeling) Islands",
    19: "Colombia", 20: "Costa Rica", 21: "Croatia", 22: "Curaçao",
    23: "Czechia", 24: "Denmark", 25: "Dominican Republic",
    26: "Ecuador", 27: "Estonia", 28: "Eswatini", 29: "Faroe Islands",
    30: "Finland", 31: "France", 32: "Germany", 33: "Ghana",
    34: "Gibraltar", 35: "Greece", 36: "Greenland", 37: "Guam",
    38: "Guatemala", 39: "Hong Kong", 40: "Hungary", 41: "Iceland",
    42: "India", 43: "Ireland", 44: "Isle of Man", 45: "Israel",
    46: "Italy", 47: "Japan", 48: "Jersey", 49: "Jordan",
    50: "Kazakhstan", 51: "Kenya", 52: "Kyrgyzstan", 53: "Laos",
    54: "Latvia", 55: "Lebanon", 56: "Lesotho", 57: "Liechtenstein",
    58: "Lithuania", 59: "Luxembourg", 60: "Malta", 61: "Mexico",
    62: "Monaco", 63: "Mongolia", 64: "Montenegro", 65: "Namibia",
    66: "Nepal", 67: "Netherlands", 68: "New Zealand", 69: "Nigeria",
    70: "North Macedonia", 71: "Northern Mariana Islands", 72: "Norway",
    73: "Oman", 74: "Pakistan", 75: "Panama", 76: "Peru",
    77: "Philippines", 78: "Poland", 79: "Portugal", 80: "Puerto Rico",
    81: "Qatar", 82: "Romania", 83: "Rwanda", 84: "Réunion",
    85: "San Marino", 86: "Senegal", 87: "Serbia", 88: "Singapore",
    89: "Slovakia", 90: "Slovenia", 91: "South Africa",
    92: "South Korea", 93: "Spain", 94: "Sri Lanka", 95: "Sweden",
    96: "Switzerland", 97: "Taiwan", 98: "Thailand", 99: "Tunisia",
    100: "Turkey", 101: "US Virgin Islands", 102: "Uganda",
    103: "Ukraine", 104: "United Arab Emirates", 105: "United States",
    106: "Vietnam", 107: "indo", 108: "malay", 109: "russia",
    110: "uk", 111: "uruguay"
}

In [ ]:
import os
from collections import defaultdict, Counter
import torch
from torchvision import transforms
from PIL import Image
import timm

# Adjust these paths
DATASET_DIR = "/content/unzipped/testSetImageStitched"
COUNTRY_MODEL_PATH = "mid_weights.pth"  # your saved weights path
SQUARE_MODEL_PATH = "mid_weights.pth"  # your saved weights path

# Constants
IMG_WIDTH, IMG_HEIGHT = 224, 224  # match training size

# Define your transform (use your training normalization)
transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
backbone = timm.create_model('tiny_vit_21m_224', pretrained=True, num_classes=0)

# Load your CustomTinyViT model (replace with your actual class if different)
class CustomTinyViT(torch.nn.Module):

    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        in_features = backbone.num_features
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(in_features, 512),   # expand
            torch.nn.GELU(),
            torch.nn.Dropout(0.5),
            torch.nn.Linear(512, 256),          # compress
            torch.nn.GELU(),
            torch.nn.Dropout(0.5),
            torch.nn.Linear(256, num_classes)   # final logits
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.classifier(x)
        return x

# Number of classes — update to your real number
NUM_CLASSES = len(CLASS_NAMES)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomTinyViT(backbone, num_classes=NUM_CLASSES)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()

# Helper to load and preprocess image
def load_and_preprocess(img_path):
    img = Image.open(img_path).convert("RGB")
    return transform(img).unsqueeze(0)  # add batch dim

# Gather test folders (your classes)
test_class_folders = sorted([
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d))
])

correct_counts = defaultdict(int)
total_counts = defaultdict(int)
top3_preds = defaultdict(Counter)

with torch.no_grad():
    for true_class in test_class_folders:
        folder_path = os.path.join(DATASET_DIR, true_class)
        for fname in os.listdir(folder_path):
            if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue

            img_path = os.path.join(folder_path, fname)
            x = load_and_preprocess(img_path).to(device)

            outputs = model(x)
            pred_idx = outputs.argmax(dim=1).item()
            pred_class_name = CLASS_NAMES.get(pred_idx, "Unknown")

            clean_true_class = true_class.split(" (")[0].lower()
            if pred_class_name.lower() == clean_true_class:
                correct_counts[true_class] += 1
            total_counts[true_class] += 1

            top3_preds[true_class][pred_class_name] += 1

        total = total_counts[true_class]
        correct = correct_counts[true_class]
        acc = correct / total if total > 0 else 0
        print(f"{true_class} got {correct}/{total} ({acc:.2%})")

        top3 = top3_preds[true_class].most_common(3)
        print("Top 3 guesses:")
        for pred_name, count in top3:
            print(f"  {pred_name}: {count} times")
        print()

# Overall accuracy
overall_correct = sum(correct_counts.values())
overall_total = sum(total_counts.values())
if overall_total > 0:
    print(f"Overall accuracy: {overall_correct}/{overall_total} ({overall_correct/overall_total:.2%})")
else:
    print("No images found.")
